# Neuronpedia Attribution Graph Corpus — Dataset Demo

This notebook demonstrates the **Neuronpedia Attribution Graph Corpus** dataset: 140 attribution graphs from gemma-2-2b across 8 capability domains (country_capital, arithmetic, antonym, translation, code_completion, multi_hop_reasoning, rhyme, sentiment).

Each record contains:
- **Input prompt** (e.g. "The capital of Japan is")
- **Full DAG-validated circuit graph** (841–1875 nodes) with node features and link weights
- **Correctness annotations**: model_correct, difficulty, expected_answer

The script below loads the data, validates it against quality criteria, and visualizes key statistics.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# networkx — pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0', 'networkx==3.6.1')

In [ ]:
import json
import os
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

## Load Data

Load the mini demo dataset from GitHub (falls back to local file).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-582cc7-circuit-motif-spectroscopy-discovering-u/main/dataset_iter2_scale_neuronped/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
examples = data["datasets"][0]["examples"]
print(f"Loaded {len(examples)} examples from dataset '{data['datasets'][0]['dataset']}'")
print(f"Metadata: {json.dumps(data['metadata'], indent=2)}")

## Configuration

Tunable parameters for the analysis. The demo uses all available examples.

In [ ]:
# Config: how many examples to process (use all available in the mini dataset)
MAX_EXAMPLES = 3  # full dataset has 140 examples across 8 domains
DATASET_NAME = "neuronpedia_attribution_graphs_v2"

## Explore Data Structure

Show the fields and a sample record from the dataset.

In [ ]:
# Limit to MAX_EXAMPLES
examples = examples[:MAX_EXAMPLES]
print(f"Processing {len(examples)} examples\n")

# Show fields of first example (excluding the large 'output' field)
ex = examples[0]
print("Fields per example:")
for k, v in ex.items():
    if k == "output":
        print(f"  {k}: <JSON string, {len(v)} chars>")
    else:
        print(f"  {k}: {v}")

# Parse and show graph structure from the output field
graph = json.loads(ex["output"])
print(f"\nGraph structure keys: {list(graph.keys())}")
print(f"  nodes: {graph['n_nodes']} (showing {len(graph['nodes'])} in demo)")
print(f"  links: {graph['n_edges']} (showing {len(graph['links'])} in demo)")
print(f"  density: {graph['density']:.4f}")
print(f"  is_dag: {graph['is_dag']}")
if graph['nodes']:
    print(f"\nSample node: {json.dumps(graph['nodes'][0], indent=2)}")
if graph['links']:
    print(f"\nSample link: {json.dumps(graph['links'][0], indent=2)}")

## Convert Records to Output Schema

Convert each example using the same logic as the original `data.py` pipeline: extract feature types from graph nodes and assemble the exp_sel_data_out record.

In [ ]:
def convert_record(record: dict, idx: int) -> dict:
    """Convert a dataset record to output schema example."""
    prompt = record["input"]
    output_data = json.loads(record["output"])
    domain = record["metadata_fold"]

    # Collect feature types from graph nodes
    feature_types = set()
    for node in output_data.get("nodes", []):
        ft = node.get("feature_type", "")
        if ft:
            feature_types.add(ft)

    # JSON-encode output
    output_str = json.dumps(output_data, separators=(",", ":"))

    return {
        "input": prompt,
        "output": output_str,
        "metadata_fold": domain,
        "metadata_n_nodes": output_data["n_nodes"],
        "metadata_n_edges": output_data["n_edges"],
        "metadata_density": output_data["density"],
        "metadata_is_dag": output_data["is_dag"],
        "metadata_slug": output_data.get("slug", ""),
        "metadata_task_type": "graph_generation",
        "metadata_n_classes": 8,
        "metadata_row_index": idx,
        "metadata_feature_names": sorted(feature_types),
        "metadata_model_correct": record.get("metadata_model_correct", "unknown"),
        "metadata_difficulty": record.get("metadata_difficulty", "medium"),
        "metadata_expected_answer": record.get("metadata_expected_answer", ""),
        "metadata_iter": 2,
    }

# Convert all records
converted = [convert_record(ex, idx) for idx, ex in enumerate(examples)]
print(f"Converted {len(converted)} examples")
for c in converted:
    print(f"  [{c['metadata_fold']}] {c['input'][:50]} -> {c['metadata_n_nodes']} nodes, {c['metadata_n_edges']} edges")

## Validate Dataset

Run the same quality checks as the original pipeline: domain distribution, DAG validation, node statistics, metadata completeness, and difficulty/correctness distributions.

In [ ]:
def validate_dataset(examples):
    """Validate the final dataset against quality criteria."""
    print("=" * 70)
    print("VALIDATION")
    print("=" * 70)

    issues = []
    total = len(examples)
    print(f"Total examples: {total}")

    # Domain distribution
    domains = Counter(ex["metadata_fold"] for ex in examples)
    print(f"\nDomain distribution:")
    for d in sorted(domains):
        cnt = domains[d]
        flag = " ***" if cnt < 5 else ""
        print(f"  {d:<25} {cnt:>4}{flag}")

    # DAG check
    dag_count = sum(1 for ex in examples if ex["metadata_is_dag"])
    print(f"\nDAGs: {dag_count}/{total} ({dag_count / total * 100:.1f}%)")

    # Node count check
    min_nodes = min(ex["metadata_n_nodes"] for ex in examples)
    max_nodes = max(ex["metadata_n_nodes"] for ex in examples)
    print(f"Nodes: min={min_nodes}, max={max_nodes}")
    if min_nodes < 10:
        issues.append(f"Some graphs have < 10 nodes (min={min_nodes})")

    # Metadata completeness
    null_correct = sum(1 for ex in examples if not ex.get("metadata_model_correct"))
    null_diff = sum(1 for ex in examples if not ex.get("metadata_difficulty"))
    null_answer = sum(1 for ex in examples if not ex.get("metadata_expected_answer"))
    print(f"\nMetadata completeness:")
    print(f"  model_correct: {total - null_correct}/{total} non-null")
    print(f"  difficulty: {total - null_diff}/{total} non-null")
    print(f"  expected_answer: {total - null_answer}/{total} non-null")

    # Difficulty distribution
    difficulties = Counter(ex["metadata_difficulty"] for ex in examples)
    print(f"\nDifficulty distribution: {dict(sorted(difficulties.items()))}")

    # Model correctness distribution
    correctness = Counter(ex["metadata_model_correct"] for ex in examples)
    print(f"Correctness distribution: {dict(sorted(correctness.items()))}")

    # Feature types
    all_ftypes = set()
    for ex in examples:
        for ft in ex.get("metadata_feature_names", []):
            all_ftypes.add(ft)
    print(f"Feature types across dataset: {sorted(all_ftypes)}")

    if issues:
        print(f"\nISSUES: {issues}")
    else:
        print(f"\nAll validation checks passed!")

    print("=" * 70)
    return len(issues) == 0

validate_dataset(converted)

## Wrap in Output Schema

Assemble the final exp_sel_data_out structure with metadata block and dataset wrapper.

In [ ]:
METADATA_BLOCK = {
    "source": "Neuronpedia API (POST /api/graph/generate)",
    "model": "gemma-2-2b",
    "description": "Attribution graphs across 8 capability domains with correctness annotations",
    "iter": 2,
    "collection_params": {
        "nodeThreshold": 0.8,
        "edgeThreshold": 0.85,
        "maxFeatureNodes": 5000,
        "maxNLogits": 10,
        "desiredLogitProb": 0.95,
    },
    "compatible_with_iter1": True,
}

def wrap_schema(examples):
    """Wrap examples in exp_sel_data_out schema."""
    return {
        "metadata": METADATA_BLOCK,
        "datasets": [
            {
                "dataset": DATASET_NAME,
                "examples": examples,
            }
        ],
    }

result = wrap_schema(converted)
output_str = json.dumps(result, separators=(",", ":"))
size_kb = len(output_str) / 1024
domains = Counter(ex["metadata_fold"] for ex in converted)
print(f"Output schema: {len(converted)} examples, {size_kb:.1f} KB")
print(f"Domains: {len(domains)} -> {dict(sorted(domains.items()))}")

## Visualization

Visualize key dataset statistics: graph sizes, domain distribution, and a sample subgraph structure.

In [ ]:
import networkx as nx

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Plot 1: Nodes & Edges per example ---
ax = axes[0, 0]
labels = [f"{c['metadata_fold']}\n{c['input'][:20]}..." for c in converted]
nodes = [c["metadata_n_nodes"] for c in converted]
edges = [c["metadata_n_edges"] for c in converted]
x = np.arange(len(labels))
width = 0.35
ax.bar(x - width/2, nodes, width, label="Nodes", color="steelblue")
ax.bar(x + width/2, [e/50 for e in edges], width, label="Edges (/50)", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel("Count")
ax.set_title("Graph Size per Example")
ax.legend()

# --- Plot 2: Domain distribution ---
ax = axes[0, 1]
domains = Counter(c["metadata_fold"] for c in converted)
domain_names = sorted(domains.keys())
domain_counts = [domains[d] for d in domain_names]
colors = plt.cm.Set3(np.linspace(0, 1, len(domain_names)))
ax.barh(domain_names, domain_counts, color=colors)
ax.set_xlabel("Count")
ax.set_title("Domain Distribution")
for i, v in enumerate(domain_counts):
    ax.text(v + 0.05, i, str(v), va="center", fontsize=9)

# --- Plot 3: Difficulty & Correctness ---
ax = axes[1, 0]
difficulties = Counter(c["metadata_difficulty"] for c in converted)
correctness = Counter(c["metadata_model_correct"] for c in converted)
categories = list(difficulties.keys()) + list(correctness.keys())
values = list(difficulties.values()) + list(correctness.values())
cat_colors = ["#66b3ff"] * len(difficulties) + ["#99ff99"] * len(correctness)
bars = ax.bar(categories, values, color=cat_colors)
ax.set_title("Difficulty & Correctness Distribution")
ax.set_ylabel("Count")
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(val), ha="center", fontsize=9)

# --- Plot 4: Sample subgraph visualization ---
ax = axes[1, 1]
sample_graph = json.loads(converted[0]["output"])
G = nx.DiGraph()
node_ids = [n["node_id"] for n in sample_graph["nodes"]]
for n in sample_graph["nodes"]:
    G.add_node(n["node_id"], feature_type=n.get("feature_type", ""))
for link in sample_graph["links"]:
    if link["source"] in node_ids and link["target"] in node_ids:
        G.add_edge(link["source"], link["target"], weight=link["weight"])

# Color by feature type
ftype_colors = {"cross layer transcoder": "steelblue", "embedding": "gold",
                "logit": "coral", "": "gray"}
node_colors = [ftype_colors.get(G.nodes[n].get("feature_type", ""), "gray") for n in G.nodes()]
pos = nx.spring_layout(G, seed=42, k=2)
nx.draw(G, pos, ax=ax, node_color=node_colors, node_size=200,
        edge_color="lightgray", arrows=True, arrowsize=10, font_size=6,
        with_labels=False)
ax.set_title(f"Sample Subgraph: {converted[0]['metadata_fold']}\n({G.number_of_nodes()} nodes, {G.number_of_edges()} edges)")

# Add legend for feature types
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=k if k else "unknown")
                   for k, c in ftype_colors.items() if k]
ax.legend(handles=legend_elements, loc="lower right", fontsize=7)

plt.tight_layout()
plt.savefig("dataset_overview.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved: dataset_overview.png")

## Summary Table

Print a readable summary of all examples in the dataset.

In [ ]:
# Results summary table
print(f"{'='*90}")
print(f"{'Idx':>3} | {'Domain':<22} | {'Input':<30} | {'Nodes':>5} | {'Edges':>6} | {'DAG':>3} | {'Diff':<6} | {'Correct':<7}")
print(f"{'-'*90}")
for c in converted:
    print(f"{c['metadata_row_index']:>3} | {c['metadata_fold']:<22} | {c['input'][:30]:<30} | "
          f"{c['metadata_n_nodes']:>5} | {c['metadata_n_edges']:>6} | "
          f"{'Y' if c['metadata_is_dag'] else 'N':>3} | "
          f"{c['metadata_difficulty']:<6} | {c['metadata_model_correct']:<7}")
print(f"{'='*90}")

# Aggregate stats
print(f"\nAggregate Statistics:")
print(f"  Total examples: {len(converted)}")
print(f"  Domains: {len(set(c['metadata_fold'] for c in converted))}")
print(f"  Avg nodes: {np.mean([c['metadata_n_nodes'] for c in converted]):.0f}")
print(f"  Avg edges: {np.mean([c['metadata_n_edges'] for c in converted]):.0f}")
print(f"  Avg density: {np.mean([c['metadata_density'] for c in converted]):.4f}")
print(f"  DAG rate: {sum(c['metadata_is_dag'] for c in converted)/len(converted)*100:.0f}%")
print(f"  Feature types: {sorted(set(ft for c in converted for ft in c['metadata_feature_names']))}")